<a id="unit-u.46607416f559493dbb3643945ae3b1a7"></a>

<a id="configure-grafana-alerting"></a>

# Grafana ескертулерін баптау

**8-модуль: Grafana көмегімен ескерту** · [Сабақ](course.md) · [Тест](quiz08_alerting_rules.ipynb)

Бұл жұмыста хост CPU-ы жұмысбасты болып тұрса хат жіберетін ескерту құрасыз. DOG стегі CPU-ды Doris-те қалай сақтайтынын қарап, бағаланатын сұрауды жазасыз. Кейін Grafana үш бөлігін баптайсыз: қалай жіберуді шешетін байланыс нүктесі, қай ескертулер сонда баратынын шешетін саясат, Grafana нені бағалап, қашан іске қосылатынын шешетін ереже. CPU-ды жүктеп, іске қосылып шешілуін бақылайсыз; хост метрикаларын тоқтатып, дерек келмегенде Grafana әрекетін көресіз. Соңында төлем инцидентіне екінші ережені кодпен жазып, жалаушасын өшіріп инцидентті аяқтайсыз.

Grafana хаттарын DOG стегімен бірге қосылған жергілікті Mailpit-ке жібереді. Еш хат компьютеріңізден шықпайды.

Шамамен 55 минут жоспарлаңыз. Үш қадам Grafana-ны күтеді: CPU жүктелгенде 4 минуттай, метрикалар тоқтағанда 3 минуттай, соңғы ереже терезесінен сәтсіз тапсырыстар шыққанша 6 минутқа дейін. 4–6 және 11-бөлімдер веб-беттерде; айналасындағы ұяшықтарды ретімен орындаңыз. Төлем ақауы қосылған 1-модульдің DOG стегі мен демосы қажет (1-зертханалық жұмыстың 5-бөлімі).

<a id="unit-u.8bb38ccd93e143d68b031c5f39916400"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты инициализациялау

`lab.sql(...)`, `lab.shell(...)` не ескерту көмекшілерін қолданбас бұрын осы ұяшықты бір рет орындаңыз.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)

<a id="unit-u.b26c3a1e14914c4fbf0e9e230cc526d2"></a>

<a id="1-check-the-host-metrics"></a>

## 1. Хост метрикаларын тексеру

Әр 15 секундта DOG Collector Docker орындайтын машина CPU-ын өлшеп, `otel_metrics_gauge` ішіне `system.cpu.utilization` ретінде жазады. macOS не Windows Docker Desktop-та ол — Linux виртуалды машинасы, Linux-та компьютеріңіздің өзі. Collector жолдарға хост атауын (`resource_attributes` ішіндегі `host.name`) қосады, сонда ескерту қай хост жұмысбасты екенін айтады. Келесі ұяшық соңғы минуттың жолдарын күтеді.

In [ ]:
host_metrics = lab.wait_for_rows(
    """
    SELECT CAST(resource_attributes['host.name'] AS STRING) AS host,
           COUNT(DISTINCT CAST(attributes['cpu'] AS STRING)) AS cpu_cores,
           COUNT(DISTINCT timestamp) AS samples,
           MAX(timestamp) AS latest_sample
    FROM otel_metrics_gauge
    WHERE metric_name = 'system.cpu.utilization'
      AND scope_name LIKE '%hostmetricsreceiver%'
      AND resource_attributes['host.name'] IS NOT NULL
      AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE
    GROUP BY host
    """,
    title="Host CPU samples from the last minute",
    hint="The DOG Collector writes these rows every 15 seconds. Run lab.prepare_environment() in Module 1 "
         "again: it starts the DOG stack, or applies the course's Collector settings to a running stack. "
         "Then rerun this cell.",
)

<a id="unit-u.89cbfd7a5fa546ab8cd1783ba7c3113e"></a>

**Күтілетін нәтиже**

Әдетте бірден бір жол: Docker хостыңыз, мысалы Docker Desktop-та `docker-desktop`, Linux-та компьютер атауы. `cpu_cores` — хост CPU саны (Docker Desktop баптауларында бар), `samples` — әр 15 секундта бірден 3 не 4, `latest_sample` қазіргі UTC уақытынан жарты минуттан аспайды.

<a id="unit-u.d9e840c7270b48f1a59d51b80f4be7cf"></a>

<a id="2-choose-the-rows-that-measure-host-cpu"></a>

## 2. Хост CPU-ын өлшейтін жолдарды таңдау

<a id="every-source-of-systemcpuutilization"></a>

### `system.cpu.utilization` барлық дереккөзі

Басқа бағдарламалар сол атаулы метриканы береді. Келесі ұяшық соңғы екі минутта `system.cpu.utilization` хабарлаған әр көзді тізеді; кейбірі минутына бір рет қана жібереді.

In [ ]:
lab.sql("""
SELECT scope_name, scope_version, service_name,
       CAST(resource_attributes['host.name'] AS STRING) AS host,
       COUNT(DISTINCT CAST(attributes['cpu'] AS STRING)) AS cpu_cores,
       COUNT(DISTINCT CAST(attributes['state'] AS STRING)) AS states,
       COUNT(*) AS row_count
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 2 MINUTE
GROUP BY scope_name, scope_version, service_name, host
ORDER BY scope_name, service_name
""", title="Sources of system.cpu.utilization in the last two minutes");

<a id="unit-u.6614cc51ebd6448b94a4f7e80324c0b0"></a>

**Күтілетін нәтиже**

Демо қосулыда төрт жол, онсыз тек DOG Collector жолы. Демо Collector-ы мен Python сервистері минутына, DOG Collector әр 15 секундта бір рет хабарлайды, сондықтан `row_count` өзгеше:

| `scope_name` | `scope_version` | `service_name` | `host` | Дереккөз |
| --- | --- | --- | --- | --- |
| `github.com/.../hostmetricsreceiver/internal/scraper/cpuscraper` | `0.144.0` | (бос) | хостыңыз | DOG Collector хост метрикалары |
| сол атау | `0.120.1` | (бос) | NULL | Демо Collector хост метрикалары |
| `opentelemetry.instrumentation.system_metrics` | `0.51b0` | `load-generator`, `recommendation` | NULL | Демоның екі Python сервисінің жүйелік метрикалары, әрқайсысына бір жол |

Төртеуі бір ядроларды санайды: контейнерлер Docker хостының ядросын бөліседі. Тек метрика атауын сүзетін ереже сол CPU-лардың бірнеше көшірмесін араластырады; тек алғашқы көз жолдарының қай хост екенін атайды. Ескерту хостқа бір көз оқиды: хост атауы бар Collector хост метрикалары қабылдағышының жолдары (`scope_name LIKE '%hostmetricsreceiver%'`). DOG Collector атауды `environments/dog-stack/collector-host-metrics.yml` ішіндегі `resourcedetection` өңдегішімен қосады.

<a id="unit-u.190025b975db431681ac7c7cff0155d9"></a>

<a id="one-sample-of-one-core"></a>

### Бір ядроның бір үлгісі

Әр жол — соңғы 15 секундта бір ядроның бір күйде болған уақыт үлесі. Келесі ұяшық `cpu0` ең жаңа үлгісін көрсетеді.

In [ ]:
lab.sql("""
SELECT timestamp, CAST(attributes['state'] AS STRING) AS state, ROUND(value, 4) AS share_of_time
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND scope_name LIKE '%hostmetricsreceiver%'
  AND resource_attributes['host.name'] IS NOT NULL
  AND attributes['cpu'] = 'cpu0'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE
ORDER BY timestamp DESC, share_of_time DESC
LIMIT 8
""", title="The newest sample of cpu0");

<a id="unit-u.4d3177f28aba42349d785c30e7a929d3"></a>

**Күтілетін нәтиже**

`timestamp` бірдей сегіз жол, күйге бірден: `idle`, `user`, `system`, `softirq`, `wait`, `nice`, `interrupt`, `steal`. Үлестер қосындысы 1. `idle` ең үлкен, жиі 0.9 маңында; қалған уақытта ядро жұмысбасты болған.

<a id="unit-u.3700cd79e97f426b9b9d6533d6e4fed0"></a>

<a id="why-a-filter-on-the-metric-name-alone-fires-on-idle-hosts"></a>

### Тек метрика атауының сүзгісі неге бос хостқа іске қосылады

Тек `metric_name` бойынша сүзіп, әр жолды 0.98 сияқты шекпен салыстыратын ереже әр жолды CPU жұмысбастылығының өлшемі деп алады. Келесі ұяшық күй бойынша соңғы минут жолдарын және қаншасы 0.9-дан жоғары екенін санайды.

In [ ]:
lab.sql("""
SELECT CAST(attributes['state'] AS STRING) AS state,
       COUNT(*) AS row_count,
       ROUND(AVG(value), 3) AS avg_share,
       SUM(IF(value > 0.9, 1, 0)) AS rows_above_0_9
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE
GROUP BY CAST(attributes['state'] AS STRING)
ORDER BY avg_share DESC
""", title="Rows of system.cpu.utilization by state in the last minute");

<a id="unit-u.74d9d8b861174a9e8f91b8aeae11bdcf"></a>

**Күтілетін нәтиже**

`idle` орташа 0.9 үлесімен бірінші, 0.9-дан асқан дерлік әр жол — `idle`; өзге күйлерде ондай жоқ. (`irq` — Python сервистерінің interrupt атауы.) Мұндай ереже CPU бос болғандықтан іске қосылады: хост тынышырақ болса, көбірек жол өтеді. Әр жол бір көздің ядросы мен күйі болғандықтан, ескертулер қай хост жұмысбасты екенін айта алмайды.

<a id="unit-u.da54b16a540b4f34beca3e1f68ddc723"></a>

<a id="3-the-alert-query"></a>

## 3. Ескерту сұрауы

Бағаланатын сұрау осы жолдарды хостқа бір санға айналдырады:

- `attributes['state'] = 'idle'` және `1 - AVG(value)`: терезенің барлық үлгісіндегі ядролардың орташа бос үлесін 1-ден шегеру. Бұл — жұмысбасты үлес, 100-ге көбейтсе пайыз.
- `scope_name LIKE '%hostmetricsreceiver%'` және `resource_attributes['host.name'] IS NOT NULL`: 2-бөлімдегідей хостқа бір дереккөз.
- `GROUP BY host`: хостқа бір жол. Grafana әр жолдан дана жасап, мәтін бағандарымен белгілейді; сондықтан хабар хостын атайды (`host=...`).
- `timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE`: терезе бұрынғы жұмыстардай ең жаңа жолда емес, қазір аяқталады. Хост тоқтаса, жолдары терезеден шығып, сұрау соңғы үлгілерді қайта-қайта бағаламай, оған ештеңе қайтармайды. 8-бөлім Grafana әрекетін көрсетеді. Кестелер UTC сақтайды, сондықтан терезе уақыты `UTC_TIMESTAMP()` негізінде алынады.

Қазір сұрауды орындаңыз. 6-бөлімде сол SQL-ді Grafana-ға қоясыз.

In [ ]:
HOST_CPU_QUERY = """
SELECT CAST(resource_attributes['host.name'] AS STRING) AS host,
       ROUND((1 - AVG(value)) * 100, 1) AS cpu_busy_percent
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND attributes['state'] = 'idle'
  AND scope_name LIKE '%hostmetricsreceiver%'
  AND resource_attributes['host.name'] IS NOT NULL
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE
GROUP BY host
"""
lab.sql(HOST_CPU_QUERY, title="Busy share of each host over the last minute (the alert query)");

<a id="unit-u.1633b359448b4bef86f9df8c97a66464"></a>

**Күтілетін нәтиже**

Бір жол: хостыңыз және әдетте 3–20 аралығындағы `cpu_busy_percent`. Бірнеше рет қайталаңыз: демо мен Doris жұмысына сай өзгереді, бірақ 6-бөлімнің 90 шегінен едәуір төмен.

<a id="unit-u.3ff797c3d3404be3ae9c9d73802624bc"></a>

<a id="4-the-mail-server-and-a-contact-point"></a>

## 4. Пошта сервері мен байланыс нүктесі

<a id="mailpit"></a>

### Mailpit

Grafana SMTP арқылы жібереді. Курстың DOG стегінде `doris-obs-mailpit` контейнеріне жібереді: Mailpit адресіне қарамастан әр хатты қабылдап, веб-жәшікте көрсетеді. [http://127.0.0.1:28025](http://127.0.0.1:28025) адресін (не `DORIS_OBS_MAILPIT_PORT` портын) ашыңыз. Келесі ұяшық Grafana-ға `environments/dog-stack/compose.dog.yml` беретін SMTP баптауларын көрсетеді.

In [ ]:
lab.shell(
    "docker inspect --format '{{range .Config.Env}}{{println .}}{{end}}' doris-obs-grafana | grep '^GF_SMTP_'",
    title="Grafana's SMTP settings",
);

<a id="unit-u.4ed0573a53bf401cb8b7b86537dcfb29"></a>

**Күтілетін нәтиже**

Шығыс төрт баптауды тізеді: `GF_SMTP_ENABLED=true`, `GF_SMTP_HOST=doris-obs-mailpit:1025` (`doris-obs` желісіндегі Mailpit контейнері мен SMTP порты), `GF_SMTP_FROM_ADDRESS`, `GF_SMTP_FROM_NAME`. Онсыз Grafana email жібере алмайды, байланыс нүктесінің Test әрекеті SMTP қатесімен сәтсіз. Нақты сервердің адресі, пайдаланушысы, құпиясөзі сол баптауларға не `grafana.ini` ішіндегі `[smtp]` бөліміне жазылады.

<a id="unit-u.74f018535ab34bff9b853f891456505a"></a>

<a id="create-a-contact-point-and-test-it"></a>

### Байланыс нүктесін құрып тексеру

Байланыс нүктесі хабарды қалай жіберуді анықтайды: мұнда бір адреске email. Grafana ішінде:

1. [http://127.0.0.1:23000/login](http://127.0.0.1:23000/login) бетінде `admin` пайдаланушысы, `admin` құпиясөзімен кіріңіз. Grafana жаңа құпиясөз сұрайды: **Skip** таңдаңыз не жаңасын орнатыңыз. Бұл Grafana тек компьютеріңізде тыңдайды. Жұмыс ұяшықтары оны жүйеге кірмей оқиды; 10-бөлім ереже құруға әкімші құпиясөзін қолданады.
2. **Alerting** > **Contact points** ашып, **+ Create contact point** таңдаңыз.
3. **Name**: `doris-course-email`. **Integration**: әдепкі `Email`. **Addresses**: `oncall@example.com`.
4. **Test**, кейін **Send test notification** таңдаңыз. Grafana «Test alert sent.» хабарлайды. Терезені жабыңыз.
5. **Save contact point** таңдаңыз.

Кейін келесі ұяшықты орындаңыз: жәшікті Mailpit API-ымен оқиды.

In [ ]:
test_email = lab.wait_for_email(
    "TestAlert",
    hint="In the contact point form, choose Test and then Send test notification. If Grafana reports an "
         "SMTP error, check the settings that the previous cell showed, then rerun this cell.",
)

<a id="unit-u.8ad9bd90473348d6aabd31eefa64b6a5"></a>

**Күтілетін нәтиже**

`Grafana (DOG course) <grafana@example.com>` жіберушісінен `oncall@example.com` адресіне, тақырыбы `[FIRING:1] TestAlert Grafana` бір хатты кесте. Браузер жәшігінде де сол хат. Мәтінінде «Notification test» қорытындысы, `alertname=TestAlert`, `instance=Grafana` белгілері бар.

Бұл хат Grafana осы нүктеге жеткізетінін көрсетеді. Ереже дұрыс іске қосылатынын, сол бағытпен баратынын не шешілу хаты келетінін көрсетпейді. Келесі бөлімдер әрқайсысын тексереді.

<a id="unit-u.6fd1e3000454400482e1992c2fd45289"></a>

<a id="5-route-the-alert-with-a-notification-policy"></a>

## 5. Ескертуді саясатпен бағыттау

Саясат белгілер бойынша қай ескертуді қай байланыс нүктесіне, қалай топтап, қашан жіберуді шешеді. Өзгеге сәйкес емес ескерту әдепкі саясатпен `grafana-default-email` нүктесіне барады. Курсқа саясат қосыңыз:

1. **Alerting** > **Notification policies** ашыңыз. **Default policy** жанынан **+ New child policy** таңдаңыз.
2. **Matching labels**: белгі `team`, оператор `=`, мән `doris-course`.
3. **Contact point**: `doris-course-email`.
4. **Override grouping** қосыңыз. **Group by** ішінде бар `grafana_folder`, `alertname` жанына `host` қосыңыз. Әр хост жеке хабар алады.
5. **Override general timings** қосып, **Group wait** `10s`, **Group interval** `1m`, **Repeat interval** `4h` енгізіңіз. Group wait — бірге іске қосылғандар бір хатпен келуі үшін жаңа топтың алғашқы хабарына дейін күту. Group interval — шешілу сияқты топ өзгерісіне дейін күту. Әдепкі 30 секунд пен 5 минут өндіріске сай; қысқа мәндер осы жұмыста уақыт үнемдейді.
6. **Save policy** таңдаңыз. Жаңа саясат `team = doris-course`, «Delivered to doris-course-email» және енгізген топтау/уақыттарды көрсетеді.

Келесі ереже `team=doris-course` белгісін ұстайды, сондықтан осы жолмен барады. Байланыс нүктесі қалай хабарлауды, саясат қай ескертулер баратынын шешеді.

<a id="unit-u.e5f8f2633e56454e925d12fd1f6acd9b"></a>

<a id="6-create-the-alert-rule"></a>

## 6. Ескерту ережесін құру

Grafana ішінде **Alerting** > **Alert rules** ашып, **+ New alert rule** таңдаңыз. Формада бес бөлім бар.

**1. Enter alert rule name** (ереже атауын енгізу): `Host CPU busy`. Төменгі ұяшықтар осы атаумен табады.

**2. Define query and alert condition** (сұрау мен шартты анықтау)

- **A** сұрауы `Doris` дереккөзін қолданады. Редакторды **Builder** орнына **Code** етіп, **Format** мәнін `Table` қалдырыңыз; 3-бөлімнің `HOST_CPU_QUERY` SQL-ін қойыңыз. **Run query** таңдаңыз: төмендегі кестеде `{host="..."}` және жұмысбасты пайыз шығады.
- **Expressions** астында Grafana **Reduce** B және **Threshold** C қосқан. Table сұрауы хостқа бір сан береді, Grafana Reduce қажет емес деп ескертеді: B өрнегін қоқыс белгісімен жойыңыз.
- **C (Threshold)** ішінде **Input** — `A`, **IS ABOVE** қалдырып, `90` енгізіңіз. C — ескерту шарты. **Preview** таңдаңыз: хост `0` және `Normal` көрсетеді.

**3. Set evaluation behavior** (бағалау тәртібін орнату)

- **Folder**: **+ New folder** таңдап, `Doris course` құрыңыз.
- **Evaluation group and interval**: **+ New evaluation group** таңдап, `host-cpu` атауын, `30s` аралығын орнатыңыз. Grafana әр ережені 30 секунд сайын бағалайды.
- **Pending period**: `1m`. Іске қосылмас бұрын шарт бір минуттың әр бағалауында орындалуы тиіс. Шектен төмен бір бағалау Normal күйіне қайтарады, минут қайта басталады. Өндірісте қысқа шың оятпау үшін 10 минуттай ұзағырақ күтеді; мұнда іске қосылуын көруге бір минут.
- **Configure no data and error handling** ашып, әдепкілерін қалдырыңыз: **Alert state if no data or all values are null** — `No Data`, **Alert state if execution error or timeout** — `Error`. 8-бөлім біріншісін көрсетеді.

**4. Configure labels and notifications** (белгілер мен хабарландыруларды баптау)

- **Labels**: **Add labels** таңдап, `team` = `doris-course`, `severity` = `warning` қосыңыз (жаңа кілт/мәнді теріп Enter басыңыз), **Save** таңдаңыз.
- **Notifications**: **Use notification policy**, кейін **Preview routing** таңдаңыз. Дана `team = doris-course` саясатына барады, «Delivered to doris-course-email».

**5. Configure notification message** (хабар мәтінін баптау)

- **Summary**: `CPU of {{ $labels.host }} is {{ $values.A }}% busy`
- **Description**: `The CPU cores of {{ $labels.host }} have been more than 90% busy on average for at least 1 minute.`

`{{ $labels.host }}` — дана хостының белгісі, `{{ $values.A }}` — сол бағалаудағы A мәні. **Save rule and exit** таңдаңыз. Тізім `Doris course > host-cpu` және `1 normal` көрсетеді. Келесі ұяшық Grafana API-ымен ережені оқиды.

In [ ]:
lab.alert_instances("Host CPU busy");

<a id="unit-u.19f85026fedb49b4a900c257a8af37d6"></a>

**Күтілетін нәтиже**

Хостыңыз бар бір `host=...` данасы `Normal` күйінде, «CPU of docker-desktop is 8.5% busy» сияқты қорытындымен. Атауы соңғы бағалаудың ең көбі 30 секунд бұрынғы уақытын, `ok` денсаулығын береді. Сақтағаннан кейін бірден «No instances yet» болуы мүмкін: жаңа ереже топтың келесі 30 секундтық сәтінде бағаланады, ұяшықты қайталаңыз. «Grafana has no alert rule named 'Host CPU busy'» деп тоқтаса, атауын тексеріңіз.

<a id="unit-u.5c5b4b3a052a4a7fb99c7e54a0ae80c9"></a>

<a id="7-make-the-cpu-busy"></a>

## 7. CPU-ды жүктеу

Келесі ұяшық Docker хостының әр CPU-ын жұмысбасты ұстайтын контейнер бастайды: әр CPU-ға ең төмен басымдықты (`nice -n 19`) бір `yes` циклі, сондықтан Doris, Grafana, Collector-лар қажет CPU уақытын әлі алады. Желдеткіштер қосылуы мүмкін. Контейнер 5 минуттан кейін өзі тоқтайды, кейінгі ұяшық ескерту іске қосылған бойда тоқтатады.

In [ ]:
from datetime import datetime, timezone

load_started = datetime.now(timezone.utc)
lab.shell(r"""
set -euo pipefail

docker rm -f doris-obs-cpu-load >/dev/null 2>&1 || true
docker run -d --rm --name doris-obs-cpu-load busybox:1.37.0 \
  sh -c 'for cpu in $(seq "$(nproc)"); do nice -n 19 yes > /dev/null & done; sleep 300'

docker ps --filter name=doris-obs-cpu-load --format '{{.Names}} {{.Status}}'
""", title="Keep every CPU of the Docker host busy");

<a id="unit-u.3cb06eb52ab14d419b64f091bfbe30ec"></a>

**Күтілетін нәтиже**

Командалар журналы `doris-obs-cpu-load Up ...` жолымен аяқталады. Алғашында шамамен 2 MB `busybox` бейнесі жүктеледі.

<a id="unit-u.534658e6505b462f858e05b5cb81e596"></a>

<a id="watch-the-alert-fire"></a>

### Ескертудің іске қосылуын бақылау

Келесі ұяшық данасы `Alerting` болғанша әр 5 секунд сайын ережені оқып, жолдағы әр өзгерісті тізеді.

In [ ]:
lab.wait_for_alert_state(
    "Host CPU busy", "Alerting",
    hint="Check that the load runs with docker ps --filter name=doris-obs-cpu-load: it stops after 5 minutes, "
         "and the previous cell starts it again. Check also the threshold 90 and the pending period 1m of "
         "the rule.",
    timeout_seconds=300,
);

<a id="unit-u.23d1c68aff354daca781fe6554a53595"></a>

**Күтілетін нәтиже**

Екі-үш минутта өзгерістер кестесі дананы `Normal`, кейін `Pending`, кейін `Alerting` деп әр өзгерткен бағалауында көрсетеді:

- Жүктеме басталғаннан шамамен бір минутта сұраудың минуттық терезесінің көбі жұмысбасты, үлес 90%-дан асады. Өсуін көруге жаңа ұяшықта `lab.sql(HOST_CPU_QUERY)` орындаңыз.
- 90-нан жоғары алғашқы бағалау `Pending` етеді.
- Екі бағалаудан, бір минуттан кейін `Alerting`. Қорытындысы сол сәттің шамамен 95% жұмысбасты үлесін береді.

In [ ]:
firing_email = lab.wait_for_email(
    "[FIRING:1] Host CPU busy",
    after=load_started,
    hint="Check the notification policy of section 5 (matching label team = doris-course, contact point "
         "doris-course-email) and the label team=doris-course of the rule.",
)

<a id="unit-u.b0000d5bc71a4c089ac15507a5859145"></a>

**Күтілетін нәтиже**

Ескерту іске қосылғаннан шамамен 10 секунд, саясаттың топты күту уақытынан кейін, тақырыбы `[FIRING:1] Host CPU busy Doris course docker-desktop (warning doris-course)` сияқты хат: іске қосылған дана саны, саясаттың топтау белгілері және ереженің басқа белгілері. Мәтінінде «CPU of ... is ...% busy» қорытындысы, мәндер (`A=` — жұмысбасты үлес, `C=1` — шарт ақиқат), `host` бар барлық белгі, ережеге және үнсіз ету формасына сілтемелер бар.

<a id="unit-u.94de9ae6ce394a2e870d83416d49e50c"></a>

<a id="stop-the-load-and-watch-the-alert-resolve"></a>

### Жүктемені тоқтатып, шешілуін бақылау

In [ ]:
lab.shell(r"""
set -euo pipefail
load_container=$(docker ps -a --filter name='^/doris-obs-cpu-load$' --format '{{.Names}}')
if [ "$load_container" = "doris-obs-cpu-load" ]; then
  docker rm -f doris-obs-cpu-load
else
  echo "CPU load already stopped (the container removes itself after at most 5 minutes)."
fi
""", title="Stop the CPU load");
lab.wait_for_alert_state(
    "Host CPU busy", "Normal",
    hint="Check that docker ps --filter name=doris-obs-cpu-load lists no container, then rerun this cell.",
    timeout_seconds=180,
);

<a id="unit-u.055ead29c1b74c99847d60eb2865824c"></a>

**Күтілетін нәтиже**

Жүктеме дереу тоқтайды. Шамамен жарты минутта бос үлгілер терезедегі жұмысбастылықты 90%-дан төмен түсіреді; келесі бағалау `Normal` күйіне қайтарады, әдетте ұяшық басталғаннан бір минутта.

In [ ]:
resolved_email = lab.wait_for_email(
    "[RESOLVED] Host CPU busy",
    after=load_started,
    hint="Grafana notifies about the resolved alert at the next group interval of the policy, 1 minute. "
         "Rerun this cell.",
)

<a id="unit-u.2f50ee017e454f47bd5af1311d8ab2a4"></a>

**Күтілетін нәтиже**

Тақырыбы `[RESOLVED] Host CPU busy Doris course docker-desktop (warning doris-course)` сияқты хат. Grafana келесі топ аралығында жібереді: мұнда әдеттегідей ескерту бір минут ішінде шешілсе, іске қосылу хатынан бір минут кейін.

Алғашқы хатпен бірге ереже, бағыт және байланыс нүктесі бір хост пен бір жүктемеде бір рет жұмыс істегенін көрсетеді. Көп хостта не шек маңында келіп-кететін жүктемеде қалай болатынын көрсетпейді.

<a id="unit-u.9bc987aa6ecb4b92a178996f0ff28cab"></a>

<a id="8-when-the-host-stops-reporting"></a>

## 8. Хост есеп беруді тоқтатқанда

Сұрау тек соңғы минутты оқиды. Хост тоқтаса, жол қайтпайды, 90-мен салыстырар мән қалмайды. **Alert state if no data or all values are null** баптауы Grafana әрекетін шешеді. Әдепкі `No Data` кезінде ереже дерек жоқ деп, Grafana сол саясатпен `DatasourceNoData` бөлек ескертуін жібереді. Келесі ұяшықтар көрсету үшін DOG Collector-ын шамамен екі минут тоқтатады. Сол кезде хост метрикалары мен одан өтетін демо сигналдары Doris-ке жетпейді. 2-модульдің тәуелсіз collector-лары FE журналын, Doris метрикаларын және өз span-дарын тікелей Doris-ке жаза алады. Бұл CPU ережесі тек DOG хост метрикаларын таңдайды; басқа кестелер өсе берсе де олардың жоқтығы No Data үшін жеткілікті.

In [ ]:
collector_stopped = datetime.now(timezone.utc)
lab.shell("docker stop doris-obs-otel-collector", title="Stop the DOG Collector");
lab.wait_for_alert_state(
    "Host CPU busy", "NoData",
    hint="Check that docker ps --filter name=doris-obs-otel-collector lists no container, and that the rule "
         "keeps the default no data handling of section 6.",
    timeout_seconds=240,
);

<a id="unit-u.f53e6bf52d85438388009c5c30ce30db"></a>

**Күтілетін нәтиже**

Collector тоқтағаннан шамамен бір минутта соңғы үлгілері терезеден шығып, денсаулығы `nodata`, жаңа `datasource_uid=doris, ref_id=A` данасы `NoData` болады: хостты емес, дереккөз бен сұрауды атайды. `host=...` данасы сол бағалаудан кейін әлі `Normal` деп тізіліп, келесісінде жоғалады: Grafana қатары нәтижеде екі бағалау бойы жоқ дананы шешеді.

In [ ]:
no_data_email = lab.wait_for_email(
    "DatasourceNoData",
    after=collector_stopped,
    hint="Check that the rule keeps the default no data handling and the label team=doris-course.",
)

<a id="unit-u.c9a670b2f7b9472eb45bda1d1d0c5b57"></a>

**Күтілетін нәтиже**

Дерек жоқ деп хабарлағаннан шамамен 10 секунд кейін тақырыбы `[FIRING:1] DatasourceNoData Doris course (doris A Host CPU busy warning doris-course)` хат. Белгілері ережені (`rulename`), дереккөзді (`datasource_uid=doris`), сұрауды (`ref_id=A`) атайды, бірақ хост жоқ: жол жоқта хостты атау мүмкін емес, қорытынды `[no value]` көрсетеді.

Ереже сұрау мүлде ештеңе қайтармаса ғана дерек жоқ дейді. Бірнеше хосттың бірі тоқтаса, қалғаны қайтады; үнсіз хост данасы екі бағалаудан кейін хабарсыз жоғалады. Соңғы жаттығу осы жағдайға сұрау жазады.

In [ ]:
lab.shell("docker start doris-obs-otel-collector", title="Start the DOG Collector again");
lab.wait_for_alert_state(
    "Host CPU busy", "Normal",
    hint="Check that the DOG Collector runs with docker ps --filter name=doris-obs-otel-collector, rerun the "
         "cell of section 1, then rerun this cell.",
    timeout_seconds=180,
);

<a id="unit-u.619d950435154e26997e4aa06d077087"></a>

**Күтілетін нәтиже**

Хост үлгілері Doris-ке жарты минутта қайта жетеді. Келесі бағалаулар `host=...` данасын `Normal` етіп қайтарып, `NoData` данасын жояды; тек `Normal` даналар қалғанда ұяшық бітеді, әдетте екі минутта. Бірнеше минуттан кейін `[RESOLVED] DatasourceNoData` хаты келеді.

<a id="unit-u.c4c3c7388fcc4e2b93a65f3af8b74426"></a>

<a id="9-the-rule-as-code"></a>

## 9. Ережені код ретінде сақтау

Grafana ережені баптау файлдарының пішімінде экспорттайды. Келесі ұяшық ережеңізді солай көрсетеді.

In [ ]:
rule_yaml = lab.export_alert_rule("Host CPU busy")

<a id="unit-u.d7eab800087c4fe29131de6985ce0e50"></a>

**Күтілетін нәтиже**

`apiVersion: 1` арқылы басталып, `Doris course` бумасындағы `host-cpu` тобын, `interval: 30s` мәнін беретін YAML. Ережеде `data` астында сұрау (`refId: A`, `rawSql`), C шек өрнегі (`type: gt`, `90`), кейін `noDataState: NoData`, `execErrState: Error`, `for: 1m`, екі аннотация және екі белгі бар: форманың әр баптауы.

Басқа Grafana-да қайта құруға осындай файлды нұсқа бақылауында сақтаңыз. Grafana іске қосылғанда `provisioning/alerting` файлдарын оқып, ережелерін құрады; веб-бет provisioned деп көрсетіп, өңдетпейді, файл жалғыз көз болып қалады. Байланыс нүктелері мен саясат беттерінде де экспорт бар. Келесі бөлім кері жолмен осы пішімде ереже жазып, Grafana-ға жібереді.

<a id="unit-u.435adfd3e9a24f089b8414895f0c9092"></a>

<a id="10-alert-on-the-failed-orders"></a>

## 10. Сәтсіз тапсырыстарға ескерту

1-модульден бері тапсырыстардың жартысы сәтсіз болды, еш ескерту айтпады. 9-бөлім ереже — Grafana экспорттайтын құжат екенін көрсетті. Оны өзіңіз де жазып жібере аласыз; топ нұсқа бақылауында сақтап, кодтай тексере алады. Келесі ұяшықтар осылай `Failed orders` екінші ережесін құрады:

- Сұрау соңғы 5 минутта аяқталған тапсырыстардың сәтсіз үлесін пайызбен есептейді. Дүкен минутына бір-төрт тапсырыс алады, 5 минутта шамамен он; қысқа терезеде бір-екеу болып, үлес 0 мен 100 арасында секірер еді.
- Шарт — 10-нан жоғары, 1 минут бойы орындалуы керек.
- 5 минутта тапсырыс аяқталмаса, үлес NULL, сұрауда мән жоқ. Бұл ереженің `noDataState` мәні — `OK`: тапсырысы жоқ дүкенде сәтсіз тапсырыс жоқ. Мүлде тапсырыс тоқтауы — жаттығудағыдай басқа ереженің сұрағы.
- `team=doris-course`, `severity=critical` белгілері 5-бөлім саясатымен 4-бөлім нүктесіне бағыттайды.

Алдымен сұрауды тексеріңіз:

In [ ]:
FAILED_ORDERS_QUERY = """
SELECT ROUND(100 * SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) / COUNT(*), 1) AS failed_percent
FROM otel_traces
WHERE service_name = 'checkout'
  AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
  AND end_time >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
"""
lab.sql(FAILED_ORDERS_QUERY, title="Share of failed orders among the orders of the last 5 minutes (the alert query)");

<a id="unit-u.9cfeb27482be4b3ea583b5d997d0b4f4"></a>

**Күтілетін нәтиже**

Бір `failed_percent` жолы, әдетте 30–70: жартысына жуығы сәтсіз, он тапсырыста кездейсоқтыққа орын бар. Демо тоқтаса, мән NULL.

<a id="unit-u.fc1ccdacb46d4e53a22b31177a7e5074"></a>

Келесі ұяшық ережені Grafana provisioning API пішімінде, 9-бөлім YAML-ының JSON нұсқасы ретінде жазады: `doris` дереккөзіндегі `A` сұрауы, `C` шегі, күту кезеңі, дерексіз/қате өңдеуі, белгілер мен аннотациялар. `lab.provision_alert_rule_group(...)` `Doris course` бумасын тауып, осы ереже мен 30 секунд аралығы бар `orders` тобын HTTP `PUT` арқылы жібереді; кейін топта дәл жіберілгендер қалады. Тұрақты `uid` қайталағанда сол ережені жаңартқызады. Жазуға `admin` пайдаланушысының құпиясөзі керек: 4-бөлімде жаңасын қоймасаңыз — `admin`.

In [ ]:
GRAFANA_ADMIN_PASSWORD = "admin"  # change it if you set a new password for admin in section 4

FAILED_ORDERS_RULE = {
    "uid": "failed-orders",
    "title": "Failed orders",
    "condition": "C",
    "data": [
        {
            "refId": "A",
            "datasourceUid": "doris",
            "relativeTimeRange": {"from": 600, "to": 0},
            "model": {"refId": "A", "datasource": {"type": "mysql", "uid": "doris"},
                      "editorMode": "code", "format": "table", "rawQuery": True, "rawSql": FAILED_ORDERS_QUERY},
        },
        {
            "refId": "C",
            "datasourceUid": "__expr__",
            "relativeTimeRange": {"from": 0, "to": 0},
            "model": {"refId": "C", "datasource": {"type": "__expr__", "uid": "__expr__"},
                      "type": "threshold", "expression": "A",
                      "conditions": [{"evaluator": {"type": "gt", "params": [10]}}]},
        },
    ],
    "for": "1m",
    "noDataState": "OK",
    "execErrState": "Error",
    "labels": {"team": "doris-course", "severity": "critical"},
    "annotations": {
        "summary": "{{ $values.A }}% of the orders of the last 5 minutes failed",
        "description": "More than 10% of the checkout service's orders failed in the last 5 minutes.",
    },
}
lab.provision_alert_rule_group("Doris course", "orders", [FAILED_ORDERS_RULE],
                               interval_seconds=30, admin_password=GRAFANA_ADMIN_PASSWORD);

<a id="unit-u.e6b7640595ac4d64b970e3916cdb52fc"></a>

**Күтілетін нәтиже**

Құпиясөзсіз екі `curl` сұрауы, кейін бір жол: `failed-orders` uid-лі `Failed orders`, `1m` күту кезеңі және екі белгі. Grafana **Alerting** > **Alert rules** ішінде `Doris course` бумасын `host-cpu`, `orders` екі тобымен көрсетеді. Ереже provisioned деп белгіленбеген, веб-бетте өңдей аласыз; файлдан жүктелгені онда тек оқылады. «Grafana refused the admin password» деп тоқтаса, `GRAFANA_ADMIN_PASSWORD` мәнін өз құпиясөзіңізге қойып қайталаңыз.

<a id="unit-u.a519be219d0040838989977ab88830c2"></a>

Жаңа ереженің іске қосылуын және хатын күтіңіз:

In [ ]:
from datetime import datetime, timezone

firing = lab.wait_for_alert_state(
    "Failed orders", "Alerting",
    hint="Check that orders still fail: paymentFailure must be 50% in the flag page, and FAILED_ORDERS_QUERY "
         "must return more than 10. Check also the label team=doris-course of the rule.",
    timeout_seconds=180,
)
fired_at = datetime.fromisoformat(firing["since_utc"][0]).replace(tzinfo=timezone.utc)
lab.wait_for_email(
    "[FIRING:1] Failed orders",
    after=fired_at,
    hint="Check the notification policy of section 5 (matching label team = doris-course, contact point "
         "doris-course-email).",
);

<a id="unit-u.8c8acc4e87b74b2ebcd73033f809d5c2"></a>

**Күтілетін нәтиже**

Шамамен екі минутта: `(no labels)` данасы алғашқы бағалауда `Pending`, бір минут кейін `Alerting`, «42.9% of the orders of the last 5 minutes failed» сияқты қорытындымен. Сұрау хостқа жол емес, бір сан береді, сондықтан өз белгілері жоқ бір дана бар. `since_utc` — `Alerting` болған уақыты; ұяшық содан кейінгі хатты күтеді. Тақырыбы `[FIRING:1] Failed orders Doris course (critical doris-course)`; іске қосылғаннан 10 секундтай кейін қорытынды, үлес `A=`, `C=1` және белгілермен келеді. Ереже тұтынушы көретін сәтсіз тапсырысты, CPU ережесі ықтимал себеп — жұмысбасты хостты бақылайды. Мұнда хост ешқашан себеп болған жоқ.

<a id="unit-u.40eb77cb180a461fbb8ade119104d781"></a>

<a id="11-end-the-incident"></a>

## 11. Инцидентті аяқтау

Тексеру аяқталды. 1-модульден бері payment төлем сұрауларының жартысына жуығын `Payment request failed. Invalid token. app.loyalty.level=gold` қатесімен қабылдамады, checkout тапсырыстарын сәтсіз етті. Дүкен тұтынушыға ештеңе көрсетпеді, сервистер қате журналын жібермеді, хост бос тұрды. Нақты жүйеде түзету payment-ке енгізілер еді; мұнда себеп — 1-модульде қосқан мүмкіндік жалаушасы. Өшіріңіз:

1. [http://127.0.0.1:28082/feature](http://127.0.0.1:28082/feature) бетін (не `DORIS_OBS_DEMO_PORT` портын) ашыңыз.
2. **paymentFailure** карточкасында **off** таңдаңыз.
3. Жоғарғы **save** басыңыз.

Келесі ұяшық flagd жаңа мән бергенше күтеді:

In [ ]:
lab.wait_for_feature_flag("paymentFailure", "off");

<a id="unit-u.bc3317aeaacf41668ec2edc3b7bf8e36"></a>

**Күтілетін нәтиже**

`off` нұсқасымен `paymentFailure` бір жолы. Енді әр төлем сәтті. **save** баспай орындасаңыз, үш минутқа дейін күтеді.

<a id="unit-u.a5c3b965d6434adc9df8d5899a100dfc"></a>

Шешілуін бақылаңыз. Ереже терезесінде соңғы 5 минуттың сәтсіз тапсырыстары әлі бар, үлес олар шыққан сайын ғана түседі:

In [ ]:
from datetime import datetime, timezone

flag_off = datetime.now(timezone.utc)
lab.wait_for_alert_state(
    "Failed orders", "Normal",
    hint="Check that paymentFailure is off (the previous cell). The last failed order leaves the 5-minute "
         "window at most 5 minutes after the flag was turned off; rerun this cell.",
    timeout_seconds=600,
)
lab.wait_for_email(
    "[RESOLVED] Failed orders",
    after=flag_off,
    hint="Grafana notifies about the resolved alert at the next group interval of the policy, 1 minute. "
         "Rerun this cell.",
);

<a id="unit-u.ca7e44f2283a48c2b683911b488989dc"></a>

**Күтілетін нәтиже**

Терезеде сәтсіз тапсырыс барда дана `Alerting` болып, үлесі 10% не аз болған алғашқы бағалауда, шамамен 5 минутта `Normal` қайтады: әдетте соңғы сәтсіз тапсырыс шығып, үлес 0 болғанда. Кері жолда күту кезеңі жоқ. Ең көбі бір минут кейін келесі топ аралығында `[RESOLVED] Failed orders Doris course (critical doris-course)` хаты келеді. Қорытындысы шешкен бағалаудың үлесін, мәндері `A=0 C=0` көрсетеді: шарт жалған.

<a id="unit-u.ab62e363b22b4c6eb4b40ba0a6e00ce3"></a>

Жалаушаны өшіргелі дүкен алған тапсырыстарды тексеріңіз:

In [ ]:
lab.sql(f"""
SELECT IF(status_code = 'STATUS_CODE_ERROR', 'failed', 'succeeded') AS outcome,
       COUNT(*) AS orders,
       MAX(timestamp) AS latest_order
FROM otel_traces
WHERE service_name = 'checkout'
  AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
  AND timestamp >= '{flag_off:%Y-%m-%d %H:%M:%S}'
GROUP BY outcome
""", title="Orders since the flag was turned off");

<a id="unit-u.1c333f406072461ba33957cb82ed7d2b"></a>

**Күтілетін нәтиже**

Соңғы минуттар тапсырыстары бар бір `succeeded` жолы, `failed` жоқ. Инцидент аяқталды.

Өткеніне қарасақ: 1-модуль ақауды бастап, трассадан көрді; 2-модуль әр сигналдағы дәлелін және қай сервис жібермейтінін тапты; 3-модуль Grafana-да бастауын тауып, хостты себептен алып тастады; 4–7-модульдер санын ықшам кестелерде сақтап, іздеуді аз оқытты, SQL-мен өлшеді; бұл модуль ескерту жасап, себебі кеткенде шешілгенін көрді.

<a id="unit-u.e3047c6faf174c44bf77dbcb51055828"></a>

<a id="12-independent-exercise"></a>

## 12. Өзіндік жаттығу

8-бөлім бір хост есеп беруді тоқтатса, басқалары хабарлап тұрғанда байқалмайтынын көрсетті. Соңғы сағатта метрикалары келген әр хост үшін соңғы үлгісінен бергі секундтарды қайтаратын сұрау жазыңыз: ереже 120-дан асса ескерте алар еді. Ескерту сұрауының дереккөз сүзгілерін қолданыңыз. Бір жол күтіңіз: хостыңыз, бірнеше секундтан шамамен жарты минутқа дейін.

In [ ]:
# Your code here

<a id="unit-u.7abbbb36718c442bbcec27fbd642d5d1"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
lab.sql("""
SELECT CAST(resource_attributes['host.name'] AS STRING) AS host,
       MAX(timestamp) AS latest_sample,
       TIMESTAMPDIFF(SECOND, MAX(timestamp), UTC_TIMESTAMP()) AS seconds_since_latest_sample
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND scope_name LIKE '%hostmetricsreceiver%'
  AND resource_attributes['host.name'] IS NOT NULL
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 HOUR
GROUP BY host
""", title="Seconds since the newest host-metrics sample of each host");

<a id="unit-u.0dc80e51db404f7093ca41490bc40b77"></a>

**Күтілетін нәтиже**

Терезе минут емес, сағат: бес минут бұрын тоқтаған хосттың әлі жолы бар, сондықтан нәтижеде `seconds_since_latest_sample` шамамен 300 жолы қалады. IS ABOVE 120 шартты ереже оны атымен ескертеді. Бір сағат үлгі келмесе, хост бұл нәтижеден де шығады; ұзағырақ үнсіздікке терезені үлкейтіңіз не есеп беруі тиіс хосттар тізімімен салыстырыңыз.

<a id="unit-u.f614fa15875844da9e8f6ccae5fdf913"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- `system.cpu.utilization` бір жолы неге хост жұмысбастылығын өлшемейтінін, сұрау оның орнына не есептейтінін түсіндіре аласыз;
- Mailpit жәшігінде сынақ хаты және `team = doris-course` саясатымен жеткізілген `Host CPU busy` іске қосылу/шешілу хаттары бар;
- күту кезеңі, топты күту және дерексіз күй баптауы көргеніңізге қалай әсер еткенін айта аласыз;
- ереженің экспортталған YAML-ында сұрау, 90 шегі және `1m` күтуі бар;
- төлем ақауы қосулыда `Failed orders` іске қосылып, өшіргенде шешілді; әрқайсысының хаты келді.

<a id="clean-up"></a>

## Тазалау

CPU жүктеме контейнері жоқ: ең көбі 5 минутта тоқтайды. `paymentFailure` жалаушасы `off` болып қалады. Екі ереже, саясат, байланыс нүктесі Grafana-да қалады, ережелер хост пен тапсырысты бақылай береді. `lab.teardown()` Grafana томымен бірге жояды, `lab.teardown(keep_data=True)` сақтайды. Қазір жою үшін алдымен **Alert rules** ережелерін, **Notification policies** ішкі саясатын, кейін **Contact points** байланыс нүктесін жойыңыз: саясат әлі қолданатын нүктені Grafana жоймайды.

<a id="unit-u.534c19c21cf54708a025eee371800b32"></a>

<a id="next"></a>

## Әрі қарай

[8-тест: Grafana көмегімен ескерту](quiz08_alerting_rules.ipynb) файлын ашыңыз. Бұл — соңғы модуль; [курс README файлы](../../README.md) бүкіл ортаны тоқтатып жоюды түсіндіреді.